In [ ]:
import pandas as pd

#Concatenação dos dfs de input
dfs = []
anos_copa = list(set(pd.read_csv('dados/copas/FIFA - World Cup Summary.csv')['YEAR']))

for ano in anos_copa:
    dfs.append(pd.read_csv(f'dados/dados_modelo/input_modelo{ano}.csv'))

df = pd.concat(dfs, ignore_index=True)
display(df)

,wins_difference,draws_difference,loses_difference,points_per_game_difference,result
0,0.166667,0.125000,-0.291667,0.62,1
1,0.000000,0.222222,-0.222222,0.22,1
2,0.444444,0.055556,0.500000,NaN,1
3,0.378788,0.090909,-0.469697,1.23,1
4,0.393333,0.155000,-0.548333,1.34,1
...,...,...,...,...,...
975,0.014406,-0.048820,0.034414,0.00,2
976,0.170204,0.035510,-0.205714,0.55,1
977,-0.007347,0.004898,0.002449,-0.02,1
978,-0.150204,-0.015510,0.165714,-0.47,1


In [ ]:
from sklearn.model_selection import cross_validate, TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np


X = df.drop(columns=['result']).fillna(0)
Y = df["result"]
#Teste usando dados de 1930 á 2022, porém sem usar ranking da fifa

#Definição do Divisor Temporal
tscv = TimeSeriesSplit(n_splits=5)

# Modelos
modelos = {
    "Logistic Regression (Normalizada)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, class_weight="balanced", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
}

#As 3 Métricas que queremos avaliar ao mesmo tempo
metricas = ['accuracy', 'roc_auc_ovr', 'neg_log_loss']

for nome, modelo in modelos.items():

    resultados = cross_validate(modelo, X, Y, cv=tscv, scoring=metricas)
    
    acc_media = np.mean(resultados['test_accuracy']) * 100
    acc_desvio = np.std(resultados['test_accuracy']) * 100 
    roc_media = np.mean(resultados['test_roc_auc_ovr'])
    logloss_media = -np.mean(resultados['test_neg_log_loss'])
    
    print(f"\n {nome}:")
    print(f"  Acurácia:  {acc_media:.2f}% (± {acc_desvio:.2f}%)")
    print(f"  ROC-AUC:   {roc_media:.4f}  (Quanto mais próximo de 1.0, melhor)")
    print(f"  Log Loss:  {logloss_media:.4f} (Quanto mais próximo de 0.0, melhor)")


 Logistic Regression (Normalizada):
  Acurácia:  49.45% (± 9.74%)
  ROC-AUC:   0.5572  (Quanto mais próximo de 1.0, melhor)
  Log Loss:  1.1936 (Quanto mais próximo de 0.0, melhor)

 Random Forest:
  Acurácia:  45.40% (± 4.78%)
  ROC-AUC:   0.5430  (Quanto mais próximo de 1.0, melhor)
  Log Loss:  1.8224 (Quanto mais próximo de 0.0, melhor)

 Gradient Boosting:
  Acurácia:  50.06% (± 8.94%)
  ROC-AUC:   0.5471  (Quanto mais próximo de 1.0, melhor)
  Log Loss:  1.5254 (Quanto mais próximo de 0.0, melhor)


In [ ]:
"""Os rankings da fifa não foram usados, e eles são um ótimo meio 
de avaliação. Eles surgiram em 1994, justamente na época 'moderna' do futebol...
Iremos testar se usar os dados de 94 em diante, porém com os dados dos rankings da fifa,
servirão para aumentar a precisão do modelo..."""

In [19]:
import pandas as pd
import numpy as np
from normalize_countries import normalize_team_name

# Para isso, teremos que refazer os csv novamente
anos_copa = [i for i in range(1994, 2023, 4)]
dfs = []

for ano in anos_copa:
    df_confrontos = pd.read_csv(f"dados/confrontos_copas/confrontos{ano}.csv")
    df_ciclo = pd.read_csv(f"dados/estatisticas_ciclos/estatisticas_ciclo{ano}.csv")
    df_ranking = pd.read_csv(f"dados/rankings/fifa_pre_copa/fifa_ranking_pre_copa_{ano}.csv")
    
    # Preparar Ranking
    df_ranking["country_full"] = df_ranking["country_full"].apply(normalize_team_name)
    df_ranking = df_ranking.set_index("country_full")
    
    # Preparar Estatísticas do Ciclo
    df_ciclo["team"] = df_ciclo["team"].apply(normalize_team_name)
    df_ciclo = df_ciclo.set_index("team")
    df_ciclo["win_rate"] = (df_ciclo["wins"] / df_ciclo["games_played"]).fillna(0)
    df_ciclo["draw_rate"] = (df_ciclo["draws"] / df_ciclo["games_played"]).fillna(0)
    df_ciclo["lose_rate"] = (df_ciclo["loses"] / df_ciclo["games_played"]).fillna(0)
    
    df_confrontos.columns = df_confrontos.columns.str.lower().str.replace(' ', '_')
    
    wins_diff, draws_diff, loses_diff, ppg_diff, rank_diff, results = [], [], [], [], [], []
    
    for row in df_confrontos.itertuples():
        home_team = normalize_team_name(row.home_team_name)
        away_team = normalize_team_name(row.away_team_name)
        
        # Pular se não tivermos os dados das estatísticas do ciclo
        if home_team not in df_ciclo.index or away_team not in df_ciclo.index:
            continue
            
        #Posição ruim caso fora do ranking
        rank_home = df_ranking.loc[home_team, "rank"] if home_team in df_ranking.index else 100
        rank_away = df_ranking.loc[away_team, "rank"] if away_team in df_ranking.index else 100
    
        wins_diff.append(df_ciclo.loc[home_team]["win_rate"] - df_ciclo.loc[away_team]["win_rate"])
        draws_diff.append(df_ciclo.loc[home_team]["draw_rate"] - df_ciclo.loc[away_team]["draw_rate"])
        loses_diff.append(df_ciclo.loc[home_team]["lose_rate"] - df_ciclo.loc[away_team]["lose_rate"])
        ppg_diff.append(df_ciclo.loc[home_team]["points_per_game"] - df_ciclo.loc[away_team]["points_per_game"])
        
        # Diferença de Ranking: Um valor NEGATIVO é bom para a equipa da Casa.
        # Exemplo: Brasil(1) - Japão(30) = -29. 
        rank_diff.append(rank_home - rank_away)
        results.append(row.resultado)
        
    # Montar o DataFrame do Ano
    df_ano = pd.DataFrame({
        "wins_difference": wins_diff,
        "draws_difference": draws_diff,
        "loses_difference": loses_diff,
        "points_per_game_difference": ppg_diff,
        "ranking_difference": rank_diff, # <--- A NÓS MÁGICA AQUI!
        "result": results
    })
    
    dfs.append(df_ano)
    print(f"Ano {ano}: Processado com sucesso! ({len(df_ano)} jogos)")


df = pd.concat(dfs, ignore_index=True)
df = df.fillna(0)
print(f"\n Jogos de 1994-2022, portando ranking da FIFA: {len(df)}")
display(df)


Ano 1994: Processado com sucesso! (52 jogos)
Ano 1998: Processado com sucesso! (64 jogos)
Ano 2002: Processado com sucesso! (64 jogos)
Ano 2006: Processado com sucesso! (64 jogos)
Ano 2010: Processado com sucesso! (64 jogos)
Ano 2014: Processado com sucesso! (80 jogos)
Ano 2018: Processado com sucesso! (64 jogos)
Ano 2022: Processado com sucesso! (64 jogos)

 Jogos de 1994-2022, portando ranking da FIFA: 516


,wins_difference,draws_difference,loses_difference,points_per_game_difference,ranking_difference,result
0,-0.005348,-0.060606,0.065954,-0.07,-32.0,0
1,0.380263,-0.113158,-0.267105,1.03,-42.0,1
2,-0.273913,0.075776,0.198137,-0.75,11.0,0
3,0.001838,-0.011029,0.009191,0.00,-10.0,2
4,-0.150719,0.310819,-0.160100,-0.14,10.0,2
...,...,...,...,...,...,...
511,0.014406,-0.048820,0.034414,0.00,1.0,2
512,0.170204,0.035510,-0.205714,0.55,-9.0,1
513,-0.007347,0.004898,0.002449,-0.02,-18.0,1
514,-0.150204,-0.015510,0.165714,-0.47,-10.0,1


In [20]:
from sklearn.model_selection import cross_validate, TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

#É literalmente o mesmo código de antes, mas o df agora é diferente

X = df.drop(columns=['result']).fillna(0)
Y = df["result"]
#Teste usando dados de 1994 á 2022, USANDO RANKING DA FIFA

#Definição do Divisor Temporal
tscv = TimeSeriesSplit(n_splits=5)

# Modelos
modelos = {
    "Logistic Regression (Normalizada)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, class_weight="balanced", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
}

#As 3 Métricas que queremos avaliar ao mesmo tempo
metricas = ['accuracy', 'roc_auc_ovr', 'neg_log_loss']

for nome, modelo in modelos.items():

    resultados = cross_validate(modelo, X, Y, cv=tscv, scoring=metricas)
    
    acc_media = np.mean(resultados['test_accuracy']) * 100
    acc_desvio = np.std(resultados['test_accuracy']) * 100 
    roc_media = np.mean(resultados['test_roc_auc_ovr'])
    logloss_media = -np.mean(resultados['test_neg_log_loss'])
    
    print(f"\n {nome}:")
    print(f"  Acurácia:  {acc_media:.2f}% (± {acc_desvio:.2f}%)")
    print(f"  ROC-AUC:   {roc_media:.4f}  (Quanto mais próximo de 1.0, melhor)")
    print(f"  Log Loss:  {logloss_media:.4f} (Quanto mais próximo de 0.0, melhor)")

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
TypeError: float() argument must be a string or a real number, not 'Series'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/pedromarques/Projetos/Modelo-preditivo---Copa-do-mundo/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 856, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/pedromarques/Projetos/Modelo-preditivo---Copa-do-mundo/.venv/lib/python3.12/site-packages/sklearn/base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/pedromarques/Projetos